# Phase B: Distribution Shift Detection trên SAM3 embeddings (multi-seed ×5)

**Goal:** Calibrate shift detector $\delta_t$ cho SA-ACI (Phase C/D), với CI mean±std.

**Outputs:**
- $\delta_t$ values cho 3 detectors × 11 conditions, mean ± std qua 5 seeds
- Calibrated $\lambda$ for SA-ACI: $\gamma_t = \gamma_0(1 + \lambda \delta_t)$

**Setup:**
- Reference: PanNuke Fold 1 (in-domain reference)
- Test conditions: Fold 2/3, simulated HED/blur/HSV shifts
- Independent of Phase A2 — only needs frozen SAM3 backbone
- Seeds [0,1,2,3,4]: re-sample subset + re-randomize augmentation per seed

**Compute:** ~5h trên T4 (embedding extraction ×5 là bottleneck). Per-seed cache → resume.

## 00 — Setup

In [ ]:
import subprocess, sys, os, platform, time, json
import numpy as np
import torch
from PIL import Image
print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__, "| CUDA:", torch.cuda.is_available())

WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/sam3_research"
SAM3_DIR = f"{REPO_DIR}/sam3"
CHECKPOINT_PATH = "/kaggle/input/datasets/hipinhththu/sam3-native-pt/sam3.pt"
DATA_ROOT = "/kaggle/input/datasets/hipinhththu/pannuke"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone",
                    "https://github.com/duonguwu/sam3_research.git", REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", SAM3_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-image", "scikit-learn", "matplotlib", "opencv-python",
                "pycocotools", "einops"], check=True)

result = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True,
)
disk_np = result.stdout.strip()
print(f"OK setup.")
print(f"  Numpy on disk (after install): {disk_np}")
print(f"  Numpy loaded in this kernel  : {np.__version__}")

if disk_np.split(".")[0] != np.__version__.split(".")[0]:
    print()
    print("=" * 65)
    print("WARNING: numpy on disk != loaded version (major mismatch).")
    print("Cell tiep theo se crash 'numpy.dtype size changed'.")
    print(">>> ACTION: Menu -> Run -> Restart & Run All (or Factory Reset)")
    print("=" * 65)

## Helper modules

In [ ]:
%%writefile pannuke_loader.py
from __future__ import annotations
from pathlib import Path
from typing import Iterator, List, Optional, Tuple
import numpy as np

CELL_TYPES: List[str] = [
    "Neoplastic", "Inflammatory", "Connective", "Dead", "Epithelial",
]

DEFAULT_ROOT = Path("/kaggle/input/datasets/hipinhththu/pannuke")

def _to_uint8(arr: np.ndarray) -> np.ndarray:
    if arr.dtype == np.uint8:
        return arr
    if arr.max() <= 1.0:
        return (arr * 255).round().clip(0, 255).astype(np.uint8)
    return arr.clip(0, 255).astype(np.uint8)

class PanNukeFold:

    def __init__(self, root, fold: int):
        self.root = Path(root)
        self.fold = fold
        sub = f"Fold {fold}"
        f = f"fold{fold}"
        candidates = [
            self.root / f / sub,   
            self.root / sub,        
        ]
        base = next((c for c in candidates if (c / "images" / f / "images.npy").exists()),
                    None)
        if base is None:
            raise FileNotFoundError(
                f"Không tìm thấy Fold {fold} ở: " + " hoặc ".join(str(c) for c in candidates)
            )
        self.images_path = base / "images" / f / "images.npy"
        self.masks_path  = base / "masks"  / f / "masks.npy"

        
        type_candidates = [
            base / "images" / f / "types.npy",   
            base / "types" / f / "types.npy",     
            base / "types.npy",
            self.root / f / "types.npy",
            self.root / "types" / f / "types.npy",
        ]
        self.types_path = next((p for p in type_candidates if p.exists()), None)

        for p in (self.images_path, self.masks_path):
            if not p.exists():
                raise FileNotFoundError(f"Missing: {p}")

        self.images = np.load(self.images_path, mmap_mode="r")
        self.masks  = np.load(self.masks_path,  mmap_mode="r")
        if self.types_path is not None:
            self.tissue_types = np.load(self.types_path, allow_pickle=True)
        else:
            n = self.images.shape[0]
            self.tissue_types = np.array(["unknown"] * n, dtype=object)
            print(f"[Fold {fold}] WARN: types.npy không tìm thấy ở:")
            for c in type_candidates:
                print(f"  - {c}")
            print(f"[Fold {fold}] Dùng placeholder 'unknown' cho {n} ảnh.")

        assert self.images.shape[0] == self.masks.shape[0] == self.tissue_types.shape[0]
        assert self.images.shape[1:] == (256, 256, 3), f"unexpected: {self.images.shape}"
        assert self.masks.shape[1:]  == (256, 256, 6), f"unexpected: {self.masks.shape}"

    def __len__(self) -> int:
        return self.images.shape[0]

    def __getitem__(self, idx: int) -> dict:
        img = _to_uint8(np.array(self.images[idx]))
        m_all = np.array(self.masks[idx], dtype=np.int32)
        masks_per_type = m_all[..., :5].transpose(2, 0, 1)
        counts = np.array(
            [int(np.unique(masks_per_type[k]).size - 1) for k in range(5)],
            dtype=np.int32,
        )
        return {
            "image": img,
            "masks": masks_per_type,
            "counts": counts,
            "tissue": str(self.tissue_types[idx]),
            "fold": self.fold,
            "idx": idx,
        }

    def iter_samples(self, start: int = 0, end: Optional[int] = None) -> Iterator[dict]:
        end = end or len(self)
        for i in range(start, end):
            yield self[i]

def load_all_folds(root=DEFAULT_ROOT) -> Tuple[PanNukeFold, PanNukeFold, PanNukeFold]:
    root = Path(root)
    return PanNukeFold(root, 1), PanNukeFold(root, 2), PanNukeFold(root, 3)

In [ ]:
%%writefile metrics.py
from __future__ import annotations
from typing import Sequence
import numpy as np

def binary_iou(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0

def binary_dice(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    sa, sb = a.sum(), b.sum()
    return float(2 * inter) / float(sa + sb) if (sa + sb) > 0 else 0.0

def match_pred_to_gt(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray],
                     iou_thresh: float = 0.5) -> dict:
    if not pred_masks and not gt_masks:
        return {"tp": 0, "fp": 0, "fn": 0, "mean_iou": 0.0}
    if not pred_masks:
        return {"tp": 0, "fp": 0, "fn": len(gt_masks), "mean_iou": 0.0}
    if not gt_masks:
        return {"tp": 0, "fp": len(pred_masks), "fn": 0, "mean_iou": 0.0}

    iou_matrix = np.zeros((len(pred_masks), len(gt_masks)), dtype=np.float32)
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            iou_matrix[i, j] = binary_iou(pm, gm)

    matched_pred, matched_gt = set(), set()
    ious = []
    pairs = np.dstack(np.unravel_index(np.argsort(-iou_matrix.ravel()), iou_matrix.shape))[0]
    for i, j in pairs:
        if iou_matrix[i, j] < iou_thresh:
            break
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(int(i)); matched_gt.add(int(j))
        ious.append(float(iou_matrix[i, j]))

    tp = len(matched_pred)
    fp = len(pred_masks) - tp
    fn = len(gt_masks)  - len(matched_gt)
    return {"tp": tp, "fp": fp, "fn": fn,
            "mean_iou": float(np.mean(ious)) if ious else 0.0}

def panoptic_quality(stats: dict) -> dict:
    tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
    sq = stats["mean_iou"]
    denom = tp + 0.5 * fp + 0.5 * fn
    rq = tp / denom if denom > 0 else 0.0
    pq = sq * rq
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"PQ": pq, "SQ": sq, "RQ": rq, "F1": f1, "P": precision, "R": recall}

def aggregate_iou_image(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray]) -> float:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu)

def aggregate_iou_dice_image(pred_masks: Sequence[np.ndarray],
                              gt_masks: Sequence[np.ndarray]) -> tuple:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu), binary_dice(pu, gu)

def union_masks(masks: Sequence[np.ndarray], shape=(256, 256)) -> np.ndarray:
    u = np.zeros(shape, dtype=bool)
    for m in masks:
        u |= m.astype(bool)
    return u.astype(np.uint8)

class ClassWiseAccumulator:

    def __init__(self, class_names):
        self.class_names = list(class_names)
        self.tp = {c: 0 for c in self.class_names}
        self.fp = {c: 0 for c in self.class_names}
        self.fn = {c: 0 for c in self.class_names}

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray, class_name: str):
        p = pred_mask.astype(bool)
        g = gt_mask.astype(bool)
        self.tp[class_name] += int(np.logical_and(p, g).sum())
        self.fp[class_name] += int(np.logical_and(p, np.logical_not(g)).sum())
        self.fn[class_name] += int(np.logical_and(np.logical_not(p), g).sum())

    def class_iou(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = tp + fp + fn
        return float(tp) / float(denom) if denom > 0 else 0.0

    def class_dice(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = 2 * tp + fp + fn
        return float(2 * tp) / float(denom) if denom > 0 else 0.0

    def mIoU(self) -> float:
        return float(np.mean([self.class_iou(c) for c in self.class_names]))

    def mDice(self) -> float:
        return float(np.mean([self.class_dice(c) for c in self.class_names]))

    def summary(self) -> dict:
        per_class = {c: {"IoU": self.class_iou(c), "Dice": self.class_dice(c),
                          "TP": self.tp[c], "FP": self.fp[c], "FN": self.fn[c]}
                      for c in self.class_names}
        return {
            "mIoU": self.mIoU(),
            "mDice": self.mDice(),
            "per_class": per_class,
        }

class PerPromptClassAccumulator:

    def __init__(self, class_names, prompts_per_class):
        self.class_names = list(class_names)
        self.prompts_per_class = {c: list(prompts_per_class[c]) for c in self.class_names}
        
        self.accs = {}
        for c, prompts in self.prompts_per_class.items():
            for p in prompts:
                self.accs[(c, p)] = ClassWiseAccumulator([c])

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray,
               class_name: str, prompt: str):
        self.accs[(class_name, prompt)].update(pred_mask, gt_mask, class_name)

    def summary(self) -> dict:
        per_class = {}
        for c in self.class_names:
            per_prompt = []
            for p in self.prompts_per_class[c]:
                acc = self.accs[(c, p)]
                per_prompt.append({
                    "prompt": p,
                    "IoU": acc.class_iou(c),
                    "Dice": acc.class_dice(c),
                    "TP": acc.tp[c], "FP": acc.fp[c], "FN": acc.fn[c],
                })
            ious = [pp["IoU"] for pp in per_prompt]
            dices = [pp["Dice"] for pp in per_prompt]
            per_class[c] = {
                "IoU": float(np.mean(ious)),   
                "Dice": float(np.mean(dices)),
                "per_prompt": per_prompt,
            }
        mIoU = float(np.mean([per_class[c]["IoU"] for c in self.class_names]))
        mDice = float(np.mean([per_class[c]["Dice"] for c in self.class_names]))
        return {"mIoU": mIoU, "mDice": mDice, "per_class": per_class}

def bootstrap_ci(values, n_boot: int = 1000, alpha: float = 0.05, seed: int = 0):
    if len(values) == 0:
        return 0.0, 0.0
    rng = np.random.default_rng(seed)
    vals = np.asarray(values, dtype=np.float64)
    boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    lo = float(np.percentile(boots, 100 * alpha / 2))
    hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return lo, hi

In [ ]:
%%writefile shift_detector.py
from __future__ import annotations
from typing import Optional, Tuple
import numpy as np
import torch

def gaussian_kernel(x: torch.Tensor, y: torch.Tensor,
                    sigmas=(1.0, 2.0, 4.0, 8.0, 16.0)) -> torch.Tensor:
    dist = torch.cdist(x, y, p=2) ** 2  
    k = torch.zeros_like(dist)
    for s in sigmas:
        k = k + torch.exp(-dist / (2 * s ** 2))
    return k / len(sigmas)

def mmd_squared(x: torch.Tensor, y: torch.Tensor,
                sigmas=(1.0, 2.0, 4.0, 8.0, 16.0)) -> float:
    n, m = x.shape[0], y.shape[0]
    Kxx = gaussian_kernel(x, x, sigmas)
    Kyy = gaussian_kernel(y, y, sigmas)
    Kxy = gaussian_kernel(x, y, sigmas)

    
    Kxx_off = (Kxx.sum() - Kxx.diag().sum()) / max(n * (n - 1), 1)
    Kyy_off = (Kyy.sum() - Kyy.diag().sum()) / max(m * (m - 1), 1)
    Kxy_avg = Kxy.mean()

    mmd2 = Kxx_off + Kyy_off - 2 * Kxy_avg
    return float(max(mmd2.item(), 0.0))  

def wasserstein_1d_mean(x: np.ndarray, y: np.ndarray) -> float:
    try:
        from scipy.stats import wasserstein_distance
    except ImportError:
        raise RuntimeError("scipy required for Wasserstein")
    D = x.shape[1]
    wds = [wasserstein_distance(x[:, d], y[:, d]) for d in range(D)]
    return float(np.mean(wds))

def energy_distance_mean(x: np.ndarray, y: np.ndarray) -> float:
    try:
        from scipy.stats import energy_distance
    except ImportError:
        raise RuntimeError("scipy required for Energy distance")
    D = x.shape[1]
    eds = [energy_distance(x[:, d], y[:, d]) for d in range(D)]
    return float(np.mean(eds))

def _as_torch(a):
    return a.float() if torch.is_tensor(a) else torch.as_tensor(np.asarray(a)).float()

def _as_np(a):
    return a.detach().cpu().numpy() if torch.is_tensor(a) else np.asarray(a)

def compute_mmd(x, y):
    return mmd_squared(_as_torch(x), _as_torch(y))

def compute_wasserstein(x, y):
    return wasserstein_1d_mean(_as_np(x), _as_np(y))

def compute_energy(x, y):
    return energy_distance_mean(_as_np(x), _as_np(y))

@torch.no_grad()
def extract_sam3_features(model, transform, pil_imgs, device: str = "cuda",
                          pool: str = "mean",
                          desc: str = "extract") -> torch.Tensor:
    from torchvision.transforms import v2
    try:
        from tqdm import tqdm
        iterator = tqdm(pil_imgs, desc=desc, leave=False)
    except ImportError:
        iterator = pil_imgs

    feats = []
    for idx, pil in enumerate(iterator):
        image = v2.functional.to_image(pil).to(device)
        image = transform(image).unsqueeze(0)
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            backbone_out = model.backbone.forward_image(image)

        
        if "vision_features" in backbone_out:
            f = backbone_out["vision_features"]
        elif "image_features" in backbone_out:
            f = backbone_out["image_features"]
        elif "backbone_fpn" in backbone_out:
            
            f = backbone_out["backbone_fpn"][-1]
        else:
            
            f = next((v for v in backbone_out.values()
                      if isinstance(v, torch.Tensor) and v.dim() == 4), None)
            if f is None:
                raise RuntimeError(f"Cannot extract features from "
                                   f"backbone_out keys: {list(backbone_out.keys())}")

        
        if f.dim() == 4:
            f = f.mean(dim=(2, 3))
        
        
        
        feats.append(f.float().detach().cpu())

        
        
        
        del image, backbone_out, f
        if (idx + 1) % 10 == 0:
            torch.cuda.empty_cache()

    return torch.cat(feats, dim=0)

class ShiftDetector:

    def __init__(self, method: str = "mmd", device: str = "cuda"):
        assert method in ("mmd", "wasserstein", "energy"), method
        self.method = method
        self.device = device
        self.ref_feats: Optional[torch.Tensor] = None

    def fit_reference(self, model, transform, ref_pil_imgs):
        self.ref_feats = extract_sam3_features(
            model, transform, ref_pil_imgs, device=self.device
        )
        return self

    def score(self, model, transform, test_pil_imgs) -> float:
        if self.ref_feats is None:
            raise RuntimeError("Call fit_reference first")
        test_feats = extract_sam3_features(
            model, transform, test_pil_imgs, device=self.device
        )
        if self.method == "mmd":
            
            return mmd_squared(
                self.ref_feats.to(self.device).float(),
                test_feats.to(self.device).float(),
            )
        elif self.method == "wasserstein":
            return wasserstein_1d_mean(self.ref_feats.numpy(), test_feats.numpy())
        else:  
            return energy_distance_mean(self.ref_feats.numpy(), test_feats.numpy())

def apply_hed_shift(img_np: np.ndarray, severity: str = "mild") -> np.ndarray:
    try:
        from skimage.color import rgb2hed, hed2rgb
    except ImportError:
        raise RuntimeError("scikit-image required")

    scales = {"mild": (0.05, 0.05), "moderate": (0.15, 0.1), "severe": (0.3, 0.2)}
    sigma, bias = scales.get(severity, scales["mild"])

    rgb = img_np.astype(np.float32) / 255.0
    hed = rgb2hed(rgb)
    
    for c in range(3):
        hed[..., c] = hed[..., c] * (1 + np.random.normal(0, sigma)) +                      np.random.normal(0, bias)
    rgb_shift = hed2rgb(hed)
    rgb_shift = np.clip(rgb_shift * 255, 0, 255).astype(np.uint8)
    return rgb_shift

def apply_blur_shift(img_np: np.ndarray, severity: str = "mild") -> np.ndarray:
    try:
        from scipy.ndimage import gaussian_filter
    except ImportError:
        raise RuntimeError("scipy required")
    sigmas = {"mild": 1.0, "moderate": 2.5, "severe": 5.0}
    s = sigmas.get(severity, 1.0)
    return gaussian_filter(img_np, sigma=(s, s, 0)).astype(np.uint8)

def apply_hsv_jitter(img_np: np.ndarray, severity: str = "mild") -> np.ndarray:
    try:
        import cv2
    except ImportError:
        
        scales = {"mild": 0.1, "moderate": 0.25, "severe": 0.5}
        s = scales.get(severity, 0.1)
        shift = np.random.normal(0, s * 255, size=3)
        return np.clip(img_np + shift, 0, 255).astype(np.uint8)

    scales = {"mild": (10, 20), "moderate": (30, 50), "severe": (60, 100)}
    h_shift, s_shift = scales.get(severity, (10, 20))
    hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV).astype(np.int32)
    hsv[..., 0] = (hsv[..., 0] + np.random.randint(-h_shift, h_shift)) % 180
    hsv[..., 1] = np.clip(hsv[..., 1] + np.random.randint(-s_shift, s_shift), 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

In [ ]:
import sys
for p in [".", "/kaggle/working", SAM3_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from pannuke_loader import PanNukeFold, CELL_TYPES, DEFAULT_ROOT
from shift_detector import (ShiftDetector, extract_sam3_features,
                             apply_hed_shift, apply_blur_shift, apply_hsv_jitter,
                             mmd_squared, wasserstein_1d_mean, energy_distance_mean)

print("Helpers loaded")

## 01 — Build SAM3 (frozen backbone only)

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_sam3_image_model(
    device=device, eval_mode=True,
    checkpoint_path=CHECKPOINT_PATH, load_from_HF=False,
)
model.eval()

processor = Sam3Processor(model, device=device, resolution=1008)
transform = processor.transform
print(f"SAM3 ready. Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## 02 — Multi-seed config + per-seed condition builder

In [ ]:
SEEDS      = [0, 1, 2, 3, 4]
REF_SIZE   = 200
TEST_SIZE  = 200
SEVERITIES = ("mild", "moderate", "severe")
DETECTORS  = ["MMD2", "Wasserstein", "Energy"]

CACHE_DIR = f"{WORK}/phase_B_seeds"
os.makedirs(CACHE_DIR, exist_ok=True)

folds = [PanNukeFold(DEFAULT_ROOT, k) for k in (1, 2, 3)]
for i, f in enumerate(folds):
    print(f"Fold {i+1}: {len(f)} patches")

def build_conditions(seed):
    rng = np.random.RandomState(seed)
    ref_idx = rng.choice(len(folds[0]), size=REF_SIZE, replace=False)
    ref_imgs = [Image.fromarray(folds[0][int(i)]["image"]).convert("RGB") for i in ref_idx]

    conds = {}
    f2 = rng.choice(len(folds[1]), size=TEST_SIZE, replace=False)
    conds["fold2"] = [Image.fromarray(folds[1][int(i)]["image"]).convert("RGB") for i in f2]
    f3 = rng.choice(len(folds[2]), size=TEST_SIZE, replace=False)
    conds["fold3"] = [Image.fromarray(folds[2][int(i)]["image"]).convert("RGB") for i in f3]

    np.random.seed(seed)
    for sev in SEVERITIES:
        conds[f"hed_{sev}"]  = [Image.fromarray(apply_hed_shift(np.array(im), sev))  for im in ref_imgs]
    for sev in SEVERITIES:
        conds[f"blur_{sev}"] = [Image.fromarray(apply_blur_shift(np.array(im), sev)) for im in ref_imgs]
    for sev in SEVERITIES:
        conds[f"hsv_{sev}"]  = [Image.fromarray(apply_hsv_jitter(np.array(im), sev)) for im in ref_imgs]
    return ref_imgs, conds

n_conds = 2 + 3 * len(SEVERITIES)
per_seed_imgs = REF_SIZE + TEST_SIZE * 2 + REF_SIZE * 3 * len(SEVERITIES)
print(f"\nPer seed: ref={REF_SIZE} + {n_conds} conditions = {per_seed_imgs} images")
print(f"Total ({len(SEEDS)} seeds): {per_seed_imgs*len(SEEDS)} images "
      f"(~{per_seed_imgs*len(SEEDS)*1.3/60:.0f} min on T4)")

## 03 — Extract features + compute shift δ, per seed (cached)

In [ ]:
from tqdm import tqdm

def run_seed(seed):
    cache = f"{CACHE_DIR}/feats_seed{seed}.npz"
    if os.path.exists(cache):
        d = np.load(cache)
        ref_feats = torch.from_numpy(d["ref"])
        test_feats = {k[5:]: torch.from_numpy(d[k]) for k in d.keys() if k.startswith("test_")}
        print(f"[seed {seed}] cache hit ({len(test_feats)} conds)")
    else:
        ref_imgs, conds = build_conditions(seed)
        print(f"[seed {seed}] extract ref({len(ref_imgs)}) + {len(conds)} conditions...")
        ref_feats = extract_sam3_features(model, transform, ref_imgs,
                                          device=device, desc=f"s{seed}-ref")
        test_feats = {}
        for name, imgs in conds.items():
            test_feats[name] = extract_sam3_features(model, transform, imgs,
                                                     device=device, desc=f"s{seed}-{name}")
        np.savez(cache, ref=ref_feats.numpy(),
                 **{f"test_{k}": v.numpy() for k, v in test_feats.items()})
        print(f"[seed {seed}] saved {cache}")

    ref_gpu = ref_feats.to(device).float()
    ref_np  = ref_feats.numpy()
    res = {}
    for name, feats in test_feats.items():
        res[name] = {
            "MMD2":        mmd_squared(ref_gpu, feats.to(device).float()),
            "Wasserstein": wasserstein_1d_mean(ref_np, feats.numpy()),
            "Energy":      energy_distance_mean(ref_np, feats.numpy()),
        }
    return res

t0 = time.time()
per_seed_results = {}
for sd in SEEDS:
    per_seed_results[sd] = run_seed(sd)
    with open(f"{CACHE_DIR}/shift_seed{sd}.json", "w") as f:
        json.dump(per_seed_results[sd], f, indent=2)
    print(f"  seed {sd} done | {(time.time()-t0)/60:.1f} min elapsed\n")

print(f"All {len(SEEDS)} seeds done in {(time.time()-t0)/60:.1f} min")
CONDITIONS = list(per_seed_results[SEEDS[0]].keys())

## 04 — Aggregate: mean ± std across seeds

In [ ]:
agg = {}
for c in CONDITIONS:
    agg[c] = {}
    for det in DETECTORS:
        vals = np.array([per_seed_results[sd][c][det] for sd in SEEDS], dtype=float)
        agg[c][det] = {"mean": float(vals.mean()), "std": float(vals.std())}

print(f"PHASE B — shift detection (mean +/- std over {len(SEEDS)} seeds)")
print("=" * 86)
print(f"{'Condition':16s} | " + " | ".join(f"{d:>20s}" for d in DETECTORS))
print("-" * 86)
for c in CONDITIONS:
    row = f"{c:16s} | "
    row += " | ".join(
        f"{agg[c][d]['mean']:>9.4f}+/-{agg[c][d]['std']:<8.4f}" for d in DETECTORS)
    print(row)

## 05 — Visualize shift hierarchy (mean ± std)

In [ ]:
import matplotlib.pyplot as plt

groups = {
    "Within-PanNuke": ["fold2", "fold3"],
    "HED stain":     ["hed_mild", "hed_moderate", "hed_severe"],
    "Blur":          ["blur_mild", "blur_moderate", "blur_severe"],
    "HSV jitter":    ["hsv_mild", "hsv_moderate", "hsv_severe"],
}
gcolor = {"Within-PanNuke": "steelblue", "HED stain": "indianred",
          "Blur": "seagreen", "HSV jitter": "goldenrod"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, DETECTORS):
    y_pos = 0
    yticks, ytlabels = [], []
    for grp_name, conds in groups.items():
        for c in conds:
            m = agg[c][metric]["mean"]
            s = agg[c][metric]["std"]
            ax.barh(y_pos, m, xerr=s, color=gcolor[grp_name],
                    label=grp_name if c == conds[0] else None,
                    error_kw={"ecolor": "black", "capsize": 3})
            yticks.append(y_pos); ytlabels.append(c)
            y_pos += 1
        y_pos += 0.5
    ax.set_yticks(yticks); ax.set_yticklabels(ytlabels)
    ax.set_xlabel(metric); ax.set_title(f"delta_t via {metric}")
    ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{WORK}/phase_B_shift_hierarchy.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved: {WORK}/phase_B_shift_hierarchy.png")

## 06 — Sanity check: expected shift order (on means)

In [ ]:
print("Sanity (expect mild < moderate < severe within each augmentation):")
for aug in ("hed", "blur", "hsv"):
    vals = [agg[f"{aug}_{s}"]["MMD2"]["mean"] for s in SEVERITIES]
    ok = vals[0] <= vals[1] <= vals[2]
    print(f"  {aug:6s}: {vals[0]:.4f} -> {vals[1]:.4f} -> {vals[2]:.4f}  {'OK' if ok else 'FAIL'}")

print("\nExpect severe shifts > within-PanNuke (fold2/fold3):")
within = max(agg["fold2"]["MMD2"]["mean"], agg["fold3"]["MMD2"]["mean"])
for aug in ("hed", "blur", "hsv"):
    sev = agg[f"{aug}_severe"]["MMD2"]["mean"]
    print(f"  {aug:6s} severe ({sev:.4f}) > within ({within:.4f}): {'OK' if sev > within else 'CHECK'}")

In [ ]:
out = {
    "config": {
        "seeds": SEEDS, "ref_size": REF_SIZE, "test_size": TEST_SIZE,
        "ref_source": "Fold 1",
        "detectors": DETECTORS,
        "shift_types": ["HED", "blur", "HSV"],
        "severities": list(SEVERITIES),
    },
    "shift_scores_aggregate": agg,
    "shift_scores_per_seed": per_seed_results,
    "groups": groups,
}
with open(f"{WORK}/phase_B_multiseed_results.json", "w") as f:
    json.dump(out, f, indent=2)
print(f"Saved: {WORK}/phase_B_multiseed_results.json")

## 07 — Calibrate λ for SA-ACI (Phase C/D)

Target: $\gamma_t = \gamma_0(1 + \lambda \delta_t)$ với $\gamma_t \in [\gamma_0, \gamma_{max}]$.

Constraint: $\gamma_0(1 + \lambda \delta_{\max}) \le \gamma_{max}$
→ $\lambda \le (\gamma_{max}/\gamma_0 - 1) / \delta_{\max}$

In [ ]:
gamma_0 = 0.05
target_gamma_max = 0.15

mmd_means = {c: agg[c]["MMD2"]["mean"] for c in CONDITIONS}
delta_max = max(mmd_means.values())
delta_typical_severe = float(np.mean([mmd_means[f"{a}_severe"] for a in ("hed", "blur", "hsv")]))

lambda_max_safe = (target_gamma_max / gamma_0 - 1) / delta_max if delta_max > 0 else 1.0

print(f"Observed delta_max (worst shift): {delta_max:.4f}")
print(f"Typical severe shift delta      : {delta_typical_severe:.4f}")
print(f"\nSuggested SA-ACI hyperparameters:")
print(f"  gamma_0       = {gamma_0}")
print(f"  gamma_max     = {target_gamma_max}")
print(f"  lambda (safe) = {lambda_max_safe:.2f}")

calib = {
    "gamma_0": gamma_0, "gamma_max": target_gamma_max,
    "delta_max_observed": float(delta_max),
    "delta_typical_severe": float(delta_typical_severe),
    "lambda_max_safe": float(lambda_max_safe),
}
with open(f"{WORK}/phase_B_lambda_calibration.json", "w") as f:
    json.dump(calib, f, indent=2)
print(f"\nSaved: {WORK}/phase_B_lambda_calibration.json")

### Phase B PASS criteria

- **Sanity hierarchy**: severity ordering correct (mild < moderate < severe) cho cả 3 augmentations
- **Cross-condition order**: severe shifts > within-PanNuke (intuition)
- **Low std**: std nhỏ so với mean → detector ổn định qua seeds
- **3 detectors agree** (qualitatively) trên rank order

### Outputs lưu vào /kaggle/working/
- `phase_B_seeds/feats_seed*.npz` — SAM3 features per seed (cache, resume)
- `phase_B_multiseed_results.json` — δ mean±std + per-seed raw
- `phase_B_shift_hierarchy.png` — visualization với error bars
- `phase_B_lambda_calibration.json` — suggested SA-ACI hyperparams